In [1]:
import os
import pandas as pd
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from sklearn.model_selection import train_test_split

In [35]:
CSV_FILE = "./train_1.csv"
IMAGE_DIR ="./train_images/train_images"

In [37]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [38]:
class DRDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform=None):
        self.df = dataframe
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image_name = self.df.iloc[idx]["id_code"] + ".png"
        image_path = os.path.join(self.image_dir, image_name)
        image = Image.open(image_path).convert("RGB")
        label = int(self.df.iloc[idx]["diagnosis"])
        if self.transform:
            image = self.transform(image)
        return image, label

In [39]:
df = pd.read_csv(CSV_FILE)
print(df.head())

        id_code  diagnosis
0  1ae8c165fd53          2
1  1b329a127307          1
2  1b32e1d775ea          4
3  1b3647865779          0
4  1b398c0494d1          0


In [40]:
train_df = pd.read_csv("train_1.csv")
val_df = pd.read_csv("valid.csv")

In [44]:
train_dataset = DRDataset(
    train_df,
    "train_images/train_images",
    train_transform
)

val_dataset = DRDataset(
    val_df,
    "val_images/val_images",
    val_transform
)

In [45]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0
)

print("Train batches :", len(train_loader))
print("Validation batches :", len(val_loader))

Train batches : 92
Validation batches : 12


In [46]:
images, labels = next(iter(train_loader))

print("Image Batch Shape :", images.shape)
print("Labels :", labels)

Image Batch Shape : torch.Size([32, 3, 224, 224])
Labels : tensor([1, 0, 1, 0, 0, 0, 2, 2, 4, 2, 1, 1, 2, 0, 0, 0, 2, 0, 3, 0, 2, 2, 1, 0,
        0, 4, 3, 0, 1, 2, 0, 0])
